# Crop Type Classification with Sentinel-2 Imagery

This notebook builds a machine learning pipeline that classifies crop type for field plots in the Luís Eduardo Magalhães (LEM) region of western Bahia, Brazil. It combines the LEM+ ground-truth dataset from Embrapa with Sentinel-2 imagery.

## 1. Load the LEM+ dataset

LEM+ provides one polygon per field and one land-use label per month, from October 2019 to September 2020. See `data/README.md` for how to download it.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

SHAPEFILE = Path("data/raw/LEM_dataset.shp")
MONTHS = [
    "Oct_2019", "Nov_2019", "Dec_2019", "Jan_2020", "Feb_2020", "Mar_2020",
    "Apr_2020", "May_2020", "Jun_2020", "Jul_2020", "Aug_2020", "Sep_2020",
]

fields = gpd.read_file(SHAPEFILE)
print(f"{len(fields)} fields, CRS: {fields.crs}")
fields.head()

1854 fields, CRS: EPSG:4326


,id,Oct_2019,Nov_2019,Dec_2019,Jan_2020,Feb_2020,Mar_2020,Apr_2020,May_2020,Jun_2020,Jul_2020,Aug_2020,Sep_2020,Note,geometry
0,0,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,Conversion area,NaN,"POLYGON ((-45.82294 -12.14292, -45.8228 -12.14..."
1,1,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,Cerrado,NaN,"POLYGON ((-45.82389 -12.14325, -45.8176 -12.14..."
2,2,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,Pasture,NaN,"POLYGON ((-45.82461 -12.14622, -45.81824 -12.1..."
3,3,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,Coffee,NaN,"POLYGON ((-45.81202 -12.14691, -45.81194 -12.1..."
4,4,Not identified,Uncultivated soil,Uncultivated soil,Uncultivated soil,Soybean,Soybean,Soybean,Uncultivated soil,Uncultivated soil,Uncultivated soil,Uncultivated soil,Uncultivated soil,NaN,"POLYGON ((-45.83022 -12.15034, -45.83024 -12.1..."


### Validate the polygons

The rest of the pipeline depends on one valid polygon per field, so these checks stop the notebook if that does not hold. Areas are computed in a projected UTM CRS, because degrees cannot be used to measure area.

In [2]:
assert fields["id"].is_unique, "Field ids must be unique"
assert fields[MONTHS].notna().all().all(), "Every field needs a label for every month"

print("Missing geometries:", int(fields.geometry.isna().sum()))
print("Invalid geometries:", int((~fields.geometry.is_valid).sum()))

utm_crs = fields.estimate_utm_crs()
fields["area_ha"] = fields.to_crs(utm_crs).area / 10_000
print("Projected CRS:", utm_crs.name)
fields["area_ha"].describe().round(1)

Missing geometries: 0
Invalid geometries: 0


Projected CRS: WGS 84 / UTM zone 23S


count    1854.0
mean      135.3
std       135.8
min         0.1
25%        50.6
50%       101.7
75%       181.7
max      2620.5
Name: area_ha, dtype: float64

### Reshape the labels

The shapefile stores one column per month. A long table, with one row per field and month, is the format needed to join the labels with the Sentinel-2 time series later.

In [3]:
labels = fields.melt(
    id_vars="id", value_vars=MONTHS, var_name="month", value_name="label"
)
labels["month"] = pd.to_datetime(labels["month"], format="%b_%Y")

print(f"{len(labels)} field-month rows, {labels['label'].nunique()} classes")
labels.head()

22248 field-month rows, 16 classes


,id,month,label
0,0,2019-10-01,Cerrado
1,1,2019-10-01,Cerrado
2,2,2019-10-01,Pasture
3,3,2019-10-01,Coffee
4,4,2019-10-01,Not identified


In [4]:
class_by_month = (
    labels.pivot_table(index="label", columns="month", aggfunc="size", fill_value=0)
    .rename(columns=lambda month: month.strftime("%b %Y"))
)
class_by_month.loc[class_by_month.sum(axis=1).sort_values(ascending=False).index]

month,Oct 2019,Nov 2019,Dec 2019,Jan 2020,Feb 2020,Mar 2020,Apr 2020,May 2020,Jun 2020,Jul 2020,Aug 2020,Sep 2020
label,,,,,,,,,,,,
Uncultivated soil,1433,1463,1307,162,38,477,1061,949,420,480,669,1137
Soybean,0,4,126,1133,1173,712,111,4,1,1,0,0
Cerrado,161,161,159,158,157,151,149,147,144,139,138,137
Millet,0,4,5,4,5,5,8,106,406,422,332,64
Pasture,95,95,95,95,95,97,97,102,102,102,102,102
Corn,0,0,66,158,168,175,181,184,131,56,36,13
Brachiaria,6,6,7,7,7,7,11,53,207,249,250,222
Cotton,0,0,1,48,121,138,138,138,137,118,71,16
Sorghum,0,0,0,1,3,3,7,51,161,163,147,60


### How often does a field change class?

A field can hold several land uses within one season, for example soybean in summer followed by millet in winter. This matters when defining the prediction target, because a single label per season has to be derived from the monthly labels.

In [5]:
classes_per_field = fields[MONTHS].nunique(axis=1)
classes_per_field.value_counts().sort_index().rename_axis("classes in season").rename("fields")

classes in season
1    313
2    559
3    929
4     53
Name: fields, dtype: int64